In [1]:
import sys
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import geopandas as gpd
from shapely.geometry import mapping
from scipy.stats import spearmanr, pearsonr
import ts_onset_cess as ocd
import pandas as pd
from fapar_def import fapar_read
import gc
import datetime

import warnings
warnings.filterwarnings('ignore')

In [2]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"
dataf = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/data/analysed_tropomi/"

In [ ]:
Y1=2018
Y2=2024

dD_all_list = []
for Y in range(Y1,Y2+1):
    print('/Volumes/blue_wd/ESA_F4R/tropomi2_merged/TROPOMI_merged_'+str(Y)+'.nc')
    dD = xr.open_mfdataset('/Volumes/blue_wd/ESA_F4R/tropomi2_merged/TROPOMI_merged_'+str(Y)+'.nc').load()
    print(dD)
    dD = dD.where((dD.lat < 5.) & (dD.lat > -5.) & (dD.lon < 29.) & (dD.lon > 8.),drop=True)
    dD = dD['deltad']
    dD.compute()
    dD_year_list = []
    for m in range(1,13):
        try:
            mp = dD.sel(time=(dD.time.dt.month==m), drop=True)
            endday = mp.time.dt.days_in_month[0].values
            bins = [datetime.datetime(Y, m, 1),datetime.datetime(Y, m, 10),datetime.datetime(Y, m, 20),datetime.datetime(Y, m, endday)]
            mp_out = mp.groupby_bins('time', bins,labels=[datetime.datetime(Y, m, 10),datetime.datetime(Y, m, 20),datetime.datetime(Y, m, endday)]).mean()
            mp_out = mp_out.rename({'time_bins':'time'})
            dD_year_list.append(mp_out)
        except:
            print('no month - ',m,' for year - ',Y)
    del dD
    gc.collect()
    dD_year = xr.concat(dD_year_list,dim='time')
    del dD_year_list
    gc.collect()
    dD_year.to_netcdf(dataf+'dD_'+str(Y)+'_10day_eq.nc',engine='h5netcdf')
    del dD_year
    gc.collect()
    print('done - ',Y)
    
        

/Volumes/blue_wd/ESA_F4R/tropomi_merged/TROPOMI_merged_2018.nc
<xarray.Dataset> Size: 2GB
Dimensions:       (time: 443, scanline: 1289, ground_pixel: 215)
Coordinates:
  * time          (time) datetime64[ns] 4kB 2018-04-30T10:50:26 ... 2018-12-3...
  * scanline      (scanline) int32 5kB 928 929 930 931 ... 2213 2214 2215 2217
  * ground_pixel  (ground_pixel) int64 2kB 0 1 2 3 4 5 ... 210 211 212 213 214
    lat           (time, scanline, ground_pixel) float32 491MB nan nan ... nan
    lon           (time, scanline, ground_pixel) float32 491MB nan nan ... nan
Data variables:
    deltad        (time, scanline, ground_pixel) float32 491MB nan nan ... nan
    h2o_vmr       (time, scanline, ground_pixel) float32 491MB nan nan ... nan
no month -  1  for year -  2018
no month -  2  for year -  2018
no month -  3  for year -  2018
no month -  4  for year -  2018
done -  2018
/Volumes/blue_wd/ESA_F4R/tropomi_merged/TROPOMI_merged_2019.nc
<xarray.Dataset> Size: 4GB
Dimensions:       (time: 651, 